In [1]:
# Load necessary Python Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Inspecting the new updated UCMR5 files.

In [2]:
ucmr5 = pd.read_csv(
    "ucmr5-occurrence-data/UCMR5_All.txt",
    sep="\t",
    encoding="latin1",
    low_memory=False  # suppresses mixed type warning for FacilityID column
)

In [3]:
ucmr5.head()

,PWSID,PWSName,Size,FacilityID,FacilityName,FacilityWaterType,SamplePointID,SamplePointName,SamplePointType,AssociatedFacilityID,...,MRL,Units,MethodID,AnalyticalResultsSign,AnalyticalResultValue,SampleEventCode,MonitoringRequirement,Region,State,UCMR1SampleType
0,010106001,Mashantucket Pequot Water System,L,00006,MPTN WTP,GU,TP1,Entry point to Dist. System,EP,NaN,...,0.005,µg/L,EPA 533,<,NaN,SE1,AM,1,01,NaN
1,010106001,Mashantucket Pequot Water System,L,00006,MPTN WTP,GU,TP1,Entry point to Dist. System,EP,NaN,...,0.003,µg/L,EPA 533,=,0.0035,SE1,AM,1,01,NaN
2,010106001,Mashantucket Pequot Water System,L,00006,MPTN WTP,GU,TP1,Entry point to Dist. System,EP,NaN,...,0.003,µg/L,EPA 533,<,NaN,SE1,AM,1,01,NaN
3,010106001,Mashantucket Pequot Water System,L,00006,MPTN WTP,GU,TP1,Entry point to Dist. System,EP,NaN,...,0.004,µg/L,EPA 533,<,NaN,SE1,AM,1,01,NaN
4,010106001,Mashantucket Pequot Water System,L,00006,MPTN WTP,GU,TP1,Entry point to Dist. System,EP,NaN,...,0.004,µg/L,EPA 533,<,NaN,SE1,AM,1,01,NaN


In [4]:
ucmr5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1928117 entries, 0 to 1928116
Data columns (total 24 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   PWSID                    object 
 1   PWSName                  object 
 2   Size                     object 
 3   FacilityID               object 
 4   FacilityName             object 
 5   FacilityWaterType        object 
 6   SamplePointID            object 
 7   SamplePointName          object 
 8   SamplePointType          object 
 9   AssociatedFacilityID     float64
 10  AssociatedSamplePointID  float64
 11  CollectionDate           object 
 12  SampleID                 object 
 13  Contaminant              object 
 14  MRL                      float64
 15  Units                    object 
 16  MethodID                 object 
 17  AnalyticalResultsSign    object 
 18  AnalyticalResultValue    float64
 19  SampleEventCode          object 
 20  MonitoringRequirement    object 
 21  Region  

In [5]:
# Count missing values in each column
ucmr5.isna().sum()

PWSID                            0
PWSName                          0
Size                             0
FacilityID                      60
FacilityName                    60
FacilityWaterType                0
SamplePointID                    0
SamplePointName                  0
SamplePointType                  0
AssociatedFacilityID       1928117
AssociatedSamplePointID    1928117
CollectionDate                   0
SampleID                         0
Contaminant                      0
MRL                              0
Units                            0
MethodID                         0
AnalyticalResultsSign            0
AnalyticalResultValue      1872227
SampleEventCode                  0
MonitoringRequirement            0
Region                           0
State                            0
UCMR1SampleType            1928117
dtype: int64

In [6]:
# Count Unique Water Systems
len(ucmr5['PWSID'].unique())

10299

In [7]:
# Check total number of rows and columns
ucmr5.shape

(1928117, 24)

In [8]:
# Remove 3 columns with 100% missing values
ucmr5 = ucmr5.drop(columns = ['AssociatedFacilityID','AssociatedSamplePointID','UCMR1SampleType'])

In [9]:
ucmr5["SamplePointType"].describe()

count     1928117
unique          1
top            EP
freq      1928117
Name: SamplePointType, dtype: object

In [10]:
ucmr5["SamplePointType"].value_counts()

SamplePointType
EP    1928117
Name: count, dtype: int64

All records in the UCMR5 dataset have SamplePointType = "EP"

In [11]:
filtered = ucmr5[ucmr5["Contaminant"].isin(["PFOA", "PFOS"])].copy()

In [12]:
# Only PFOA and PFOS?
filtered["Contaminant"].unique()

array(['PFOS', 'PFOA'], dtype=object)

In [13]:
filtered.shape

(128400, 21)

In [14]:
# AnalyticalResultValue: missing = non-detect, present = detection
print(f"Total PFOA/PFOS samples: {len(filtered)}")
print(f"Non-detects: {filtered['AnalyticalResultValue'].isna().sum()}")
print(f"Detections: {filtered['AnalyticalResultValue'].notna().sum()}")
print(f"Detection rate: {(filtered['AnalyticalResultValue'].notna().sum() / len(filtered)) * 100:.2f}%")

Total PFOA/PFOS samples: 128400
Non-detects: 119752
Detections: 8648
Detection rate: 6.74%


In [15]:
print("\n--- AnalyticalResultsSign Distribution ---")
print(filtered['AnalyticalResultsSign'].value_counts())


--- AnalyticalResultsSign Distribution ---
AnalyticalResultsSign
<    119752
=      8648
Name: count, dtype: int64


In [16]:
filtered['MRL'].value_counts()

MRL
0.004    128400
Name: count, dtype: int64

The dataset was restricted to samples measuring PFOA and PFOS, the two PFAS compounds of interest. All other contaminants were excluded, and a new dataset containing only PFOA and PFOS measurements was created for subsequent analysis.

In [17]:
filtered.to_csv("ucmr5_pfoa_pfos.csv", index=False)